In [1]:
!pip -q install yfinance pandas numpy scikit-learn joblib

In [2]:
import pandas as pd
import yfinance as yf
from datetime import datetime, timezone

TICKER = "CEZ.PR"

END = datetime.now(timezone.utc).date()
START = (pd.Timestamp(END) - pd.DateOffset(years=10)).date()

df = yf.download(
    TICKER,
    start=str(START),
    end=str(END),
    interval="1d",
    auto_adjust=False,
    progress=False
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.reset_index()

print("Počet řádků:", len(df))
df.head()

Počet řádků: 2505


Price,Date,Adj Close,Close,High,Low,Open,Volume
0,2016-02-25,187.944550,366.000000,371.000000,366.000000,369.000000,241886
1,2016-02-26,188.971542,368.000000,371.000000,365.500000,369.500000,452045
2,2016-02-29,188.458054,367.000000,370.000000,365.700012,368.299988,795995
3,2016-03-01,187.431061,365.000000,368.000000,363.000000,368.000000,359134
4,2016-03-02,188.252670,366.600006,367.100006,365.100006,366.200012,357439


In [3]:
import numpy as np

df["return_1d"] = df["Close"].pct_change()
df["sma_5"] = df["Close"].rolling(5).mean()
df["sma_10"] = df["Close"].rolling(10).mean()
df["volatility_5"] = df["return_1d"].rolling(5).std()
df["volume_change_1d"] = df["Volume"].pct_change()

df["label"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

df = df.replace([np.inf, -np.inf], np.nan)
df = df.iloc[:-1]
df = df.dropna().reset_index(drop=True)

print("Počet řádků po úpravách:", len(df))
df.head()

Počet řádků po úpravách: 2494


Price,Date,Adj Close,Close,High,Low,Open,Volume,return_1d,sma_5,sma_10,volatility_5,volume_change_1d,label
0,2016-03-09,196.109344,381.899994,386.500000,380.000000,385.600006,343949,-0.009595,382.240002,374.380002,0.014017,-0.590998,1
1,2016-03-10,197.290405,384.200012,386.899994,380.000000,383.799988,423505,0.006023,383.680005,376.200003,0.008485,0.231302,1
2,2016-03-11,198.009338,385.600006,386.500000,382.299988,385.799988,415316,0.003644,384.380005,377.960004,0.006599,-0.019336,1
3,2016-03-14,201.809280,393.000000,395.299988,384.399994,386.899994,528858,0.019191,386.060004,380.560004,0.010260,0.273387,0
4,2016-03-15,201.244415,391.899994,391.899994,383.399994,388.500000,545456,-0.002799,387.320001,383.250003,0.010765,0.031385,1


In [4]:
FEATURES = [
    "return_1d",
    "sma_5",
    "sma_10",
    "volatility_5",
    "volume_change_1d"
]

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

X_train = train_df[FEATURES].values
y_train = train_df["label"].values

X_test = test_df[FEATURES].values
y_test = test_df["label"].values

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 1995
Test: 499


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=2000, random_state=42))])

In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.4909819639278557
Confusion matrix:
[[ 14 235]
 [ 19 231]]

Classification report:
              precision    recall  f1-score   support

           0       0.42      0.06      0.10       249
           1       0.50      0.92      0.65       250

    accuracy                           0.49       499
   macro avg       0.46      0.49      0.37       499
weighted avg       0.46      0.49      0.37       499



In [7]:
import joblib

joblib.dump(pipeline, "cez_direction_model.joblib")

['cez_direction_model.joblib']